In [ ]:
import json

entries = []
source_texts = []

with open("../data_bank.jsonl") as f:
    for line in f:
        entry = json.loads(line)
        entries.append(entry)
        source_texts.append(entry["source_text"])


In [ ]:
# Build a prompt that includes all abstracts and asks for topic classification
prompt = f"""
You are a research assistant. I will give you a list of research paper abstracts and metadata. Your task is to:
1. Identify a small set of shared high-level research **topics** (e.g., "Prompt Learning", "Reinforcement Learning", "Knowledge Distillation", etc.)
2. Assign one topic label to each item.

Format your response as a JSON list:
[
  {{ "index": 0, "topic": "Prompt Learning" }},
  {{ "index": 1, "topic": "Knowledge Distillation" }},
  ...
]

Here are the abstracts and metadata:

""" + "\n\n".join([f"{i}. {text.strip()}" for i, text in enumerate(source_texts)])


In [2]:
### Analyse the data topics

import json
from collections import Counter, defaultdict

topic_counter = Counter()
topic_examples = defaultdict(list)

with open("data_bank_harpa_with_topics.jsonl") as f:
    for line in f:
        entry = json.loads(line)
        topic = entry.get("shared_topic", "UNKNOWN")
        topic_counter[topic] += 1
        topic_examples[topic].append(entry["idea_id"])  # or entry["source_text"] if you prefer

# Print summary
print("📊 Topic Distribution:\n")
for topic, count in topic_counter.most_common():
    print(f"{topic}: {count} entries")

print("\n🔍 Total unique topics:", len(topic_counter))
print("🧾 Total entries:", sum(topic_counter.values()))


📊 Topic Distribution:

Multimodal Learning: 34 entries
Information Extraction: 30 entries
Parameter-Efficient Fine-Tuning: 29 entries
Multilingual NLP: 29 entries
Information Retrieval: 28 entries
Prompt Learning: 28 entries
Prompt Engineering: 24 entries
Text Summarization: 24 entries
Reasoning in LLMs: 22 entries
Question Answering: 20 entries
LLM Evaluation: 20 entries
In-Context Learning: 20 entries
Sentiment Analysis: 16 entries
Bias and Fairness: 16 entries
Contrastive Learning: 15 entries
Instruction Tuning: 15 entries
Retrieval-Augmented Generation: 14 entries
Hallucination and Factuality: 14 entries
Multilingual Language Models: 14 entries
AI Safety and Ethics: 13 entries
Multimodal LLMs: 13 entries
LLM Factuality: 11 entries
Evaluation Metrics: 10 entries
Machine Translation: 10 entries
Long Context Processing: 10 entries
Code Understanding and Generation: 10 entries
Model Architecture: 10 entries
Retrieval-Augmented Language Models: 10 entries
Representation Learning: 10 ent

In [ ]:
import json
import re
from pathlib import Path

# Path to your result file
results_path = Path("../smoke_test_results/batch_results.jsonl")

# Function to parse the Claude evaluation output
def parse_claude_output(text):
    def extract(tag):
        match = re.search(rf"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
        return match.group(1).strip() if match else None

    return {
        "rubric": extract("rubric"),
        "eval": extract("eval"),
        "answer": extract("answer")
    }

# Load and parse all results
parsed_results = []
with open(results_path, "r") as f:
    for line in f:
        entry = json.loads(line)
        text = entry["result"]["message"]["content"][0]["text"]
        parsed = parse_claude_output(text)
        parsed["custom_id"] = entry["custom_id"]
        parsed_results.append(parsed)

# Print the parsed contents of the first result
print("First parsed response:")
print(json.dumps(parsed_results[0], indent=2))


In [ ]:
from datasets import load_from_disk, DatasetDict
from datasets import Dataset
import hashlib
import random
import json

# ------------------ Config ------------------ #
DATASET_PATH = "rmr1_sft_dataset"
TRAIN_OUT = "trace_train.jsonl"
TEST_OUT = "trace_test.jsonl"
SKIPPED_OUT = "skipped.jsonl"
SPLIT_RATIO = 0.8
RETRIES = 20

# ------------------ Load Dataset ------------------ #
dataset = load_from_disk(DATASET_PATH)

# ------------------ Proposal Signature Function ------------------ #
def get_proposal_signature(text):
    return hashlib.md5(text.strip().encode("utf-8")).hexdigest()

# ------------------ Extract Proposal Signatures ------------------ #
def extract_signatures(example):
    content = example['context_messages'][1]['content']
    p_a = content.split('[The Start of Proposal A]')[1].split('[The End of Proposal A]')[0].strip()
    p_b = content.split('[The Start of Proposal B]')[1].split('[The End of Proposal B]')[0].strip()
    return {
        "sig_a": get_proposal_signature(p_a),
        "sig_b": get_proposal_signature(p_b),
        "full_example": example
    }

annotated = list(map(extract_signatures, dataset))

# ------------------ Prepare Unique Proposal Set ------------------ #
unique_proposals = set()
for ex in annotated:
    unique_proposals.add(ex["sig_a"])
    unique_proposals.add(ex["sig_b"])

# ------------------ Try Multiple Splits ------------------ #
best_train, best_test = [], []
min_skipped = float("inf")
all_sigs = list(unique_proposals)
split_idx = int(SPLIT_RATIO * len(all_sigs))

for _ in range(RETRIES):
    random.shuffle(all_sigs)
    train_sigs = set(all_sigs[:split_idx])
    test_sigs = set(all_sigs[split_idx:])

    train_set, test_set, skipped_set = [], [], []

    for ex in annotated:
        if ex["sig_a"] in train_sigs and ex["sig_b"] in train_sigs:
            train_set.append(ex["full_example"])
        elif ex["sig_a"] in test_sigs and ex["sig_b"] in test_sigs:
            test_set.append(ex["full_example"])
        else:
            skipped_set.append(ex["full_example"])

    if len(skipped_set) < min_skipped:
        min_skipped = len(skipped_set)
        best_train = train_set
        best_test = test_set
        best_skipped = skipped_set

# Convert to Hugging Face Datasets
train_dataset = Dataset.from_list(best_train)
test_dataset = Dataset.from_list(best_test)
skipped_dataset = Dataset.from_list(best_skipped)

# Save all splits including skipped
full_dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset,
    "skipped": skipped_dataset
})

full_dataset.save_to_disk("rmr1_sft_split_dataset")
print("💾 Hugging Face datasets (train/test/skipped) saved to 'rmr1_sft_split_dataset'")

print(f"✅ Train: {len(best_train)} | Test: {len(best_test)} | Skipped: {len(best_skipped)}")


In [1]:
from datasets import load_from_disk, DatasetDict, Dataset
import hashlib, random
import networkx as nx   # pip install networkx
from collections import Counter

SEED = 42
random.seed(SEED)

DATASET_PATH = "rmr1_sft_dataset_labels"
SPLIT_RATIO = 0.7   # ~70% clusters -> train
VAL_RATIO   = 0.1   # ~10% clusters -> val, rest -> test

# ------------------ Load Dataset ------------------ #
dataset = load_from_disk(DATASET_PATH)

def get_proposal_signature(text):
    return hashlib.md5(text.strip().encode("utf-8")).hexdigest()

def extract_signatures(example):
    content = example['context_messages'][1]['content']
    p_a = content.split('[The Start of Proposal A]')[1].split('[The End of Proposal A]')[0].strip()
    p_b = content.split('[The Start of Proposal B]')[1].split('[The End of Proposal B]')[0].strip()
    return {
        "sig_a": get_proposal_signature(p_a),
        "sig_b": get_proposal_signature(p_b),
        "full_example": example
    }

annotated = list(map(extract_signatures, dataset))

# ------------------ Build Proposal Graph ------------------ #
G = nx.Graph()
for ex in annotated:
    G.add_edge(ex["sig_a"], ex["sig_b"])

components = list(nx.connected_components(G))
random.shuffle(components)

# ------------------ Component Stats ------------------ #
print(f"🔎 Found {len(components)} connected components")

comp_sizes = [len(c) for c in components]
print("📊 Proposal count per component (summary):")
print(f"   Min: {min(comp_sizes)} | Max: {max(comp_sizes)} | Mean: {sum(comp_sizes)/len(comp_sizes):.2f}")
print("   Size distribution (top 20 sizes):", Counter(comp_sizes).most_common(20))

# also count examples per component
comp_example_counts = []
for comp in components:
    sigs = set(comp)
    num_examples = sum(1 for ex in annotated if ex["sig_a"] in sigs and ex["sig_b"] in sigs)
    comp_example_counts.append(num_examples)

print("📊 Example count per component (summary):")
print(f"   Min: {min(comp_example_counts)} | Max: {max(comp_example_counts)} | Mean: {sum(comp_example_counts)/len(comp_example_counts):.2f}")

# ------------------ Assign Components to Splits ------------------ #
n = len(components)
n_train = int(SPLIT_RATIO * n)
n_val   = int(VAL_RATIO * n)

train_clusters = components[:n_train]
val_clusters   = components[n_train:n_train+n_val]
test_clusters  = components[n_train+n_val:]

train_sigs = set().union(*train_clusters)
val_sigs   = set().union(*val_clusters)
test_sigs  = set().union(*test_clusters)

# ------------------ Place Examples ------------------ #
train_set, val_set, test_set = [], [], []
for ex in annotated:
    if ex["sig_a"] in train_sigs and ex["sig_b"] in train_sigs:
        train_set.append(ex["full_example"])
    elif ex["sig_a"] in val_sigs and ex["sig_b"] in val_sigs:
        val_set.append(ex["full_example"])
    elif ex["sig_a"] in test_sigs and ex["sig_b"] in test_sigs:
        test_set.append(ex["full_example"])
    else:
        raise RuntimeError("Example crosses splits unexpectedly!")

# ------------------ Convert to Datasets ------------------ #
train_dataset = Dataset.from_list(train_set)
val_dataset   = Dataset.from_list(val_set)
test_dataset  = Dataset.from_list(test_set)

full_dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset,
})

full_dataset.save_to_disk("rmr1_sft_labels_split_dataset")
print("💾 Saved to 'rmr1_sft_split_dataset'")
print(f"✅ Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")


/opt/miniconda3/envs/rm-r1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔎 Found 151 connected components
📊 Proposal count per component (summary):
   Min: 2 | Max: 34 | Mean: 7.73
   Size distribution (top 20 sizes): [(5, 80), (10, 14), (4, 11), (3, 6), (6, 6), (8, 5), (9, 3), (14, 3), (2, 3), (20, 3), (16, 2), (24, 2), (28, 2), (15, 2), (29, 2), (13, 2), (34, 1), (7, 1), (11, 1), (30, 1)]
📊 Example count per component (summary):
   Min: 1 | Max: 273 | Mean: 22.91


Saving the dataset (1/1 shards): 100%|██████████| 452/452 [00:00<00:00, 24339.47 examples/s]

💾 Saved to 'rmr1_sft_split_dataset'
✅ Train: 2595 | Val: 412 | Test: 452


In [24]:
from datasets import load_from_disk
import hashlib

# load your dataset
ds = load_from_disk("rmr1_sft_split_dataset")

def get_proposal_signature(text):
    return hashlib.md5(text.strip().encode("utf-8")).hexdigest()

def extract_proposals(split):
    sigs = set()
    for ex in split:
        content = ex['context_messages'][1]['content']
        p_a = content.split('[The Start of Proposal A]')[1].split('[The End of Proposal A]')[0].strip()
        p_b = content.split('[The Start of Proposal B]')[1].split('[The End of Proposal B]')[0].strip()
        sigs.add(get_proposal_signature(p_a))
        sigs.add(get_proposal_signature(p_b))
    return sigs

train_props = extract_proposals(ds["train"])
val_props   = extract_proposals(ds["validation"])
test_props  = extract_proposals(ds["test"])

overlap_train_val  = train_props.intersection(val_props)
overlap_train_test = train_props.intersection(test_props)
overlap_val_test   = val_props.intersection(test_props)

print(f"🚨 Overlap train–val: {len(overlap_train_val)}")
print(f"🚨 Overlap train–test: {len(overlap_train_test)}")
print(f"🚨 Overlap val–test: {len(overlap_val_test)}")



🚨 Overlap train–val: 0
🚨 Overlap train–test: 0
🚨 Overlap val–test: 0


In [3]:
from datasets import load_from_disk

# Load the dataset
train_ds = load_from_disk("../rlvr_preference_data_all_success_failure")

print(train_ds)
print(train_ds["train"][0])  # look at the first example


DatasetDict({
    train: Dataset({
        features: ['context_messages', 'winner', 'ground_truth', 'labels'],
        num_rows: 226170
    })
})
{'context_messages': [{'content': "\nPlease act as an impartial evaluator and assess the testability or successful execution of the two research proposals generated by an Ideator to execute in an Automated Scientific Discovery (ASD) Agent. \n\n## START of description of ASD agent:\n\nThe ASD Agent is an automated discovery system that writes Python-based experiments, executes them in containers, and analyzes results—usually across multiple independent runs with a meta-analysis.\nMore specifically, these automated scientific discovery systems operate by having code-based experimentation.  They can generate code, run it, debug it, analyze results, create reports, and so forth.\n\nWhat they can't do:\n    1. They can't run physical experiments (e.g. wet-lab experiments).\n    2. They can't perform anything that requires human involvement (e.g. a

In [ ]:
from datasets import load_from_disk

# Load the full split dataset
dataset = load_from_disk("rmr1_sft_split_dataset")
print(dataset)  # shows 'train', 'test', 'skipped'


In [16]:
from datasets import load_from_disk
import numpy as np

# Load dataset dict
ds_dict = load_from_disk("rmr1_sft_labels_split_dataset")
print(ds_dict)  # shows available splits, e.g. train/validation/test

# Select one split
df = ds_dict["test"].to_pandas()


print("\nGround truth distribution:")
print(df["ground_truth"].value_counts())

print("\nExecution label combinations:")
df["labels_tuple"] = df["labels"].apply(lambda x: tuple(x.tolist()) if isinstance(x, np.ndarray) else tuple(x))

print("\nExecution label combinations (unique counts):")
print(df["labels_tuple"].value_counts())



DatasetDict({
    train: Dataset({
        features: ['context_messages', 'winner', 'ground_truth', 'labels'],
        num_rows: 2595
    })
    validation: Dataset({
        features: ['context_messages', 'winner', 'ground_truth', 'labels'],
        num_rows: 412
    })
    test: Dataset({
        features: ['context_messages', 'winner', 'ground_truth', 'labels'],
        num_rows: 452
    })
})

Ground truth distribution:
ground_truth
A    231
B    221
Name: count, dtype: int64

Execution label combinations:

Execution label combinations (unique counts):
labels_tuple
(failure, success)      297
(failure, uncertain)     92
(success, uncertain)     63
Name: count, dtype: int64


In [17]:
from datasets import load_from_disk

ds_dict = load_from_disk("rmr1_sft_labels_split_dataset")

target_combo = ("failure", "success")
filtered_splits = {}

for split, split_ds in ds_dict.items():
    sdf = split_ds.to_pandas()
    sdf["labels_tuple"] = sdf["labels"].apply(lambda x: tuple(x.tolist()))
    filtered_df = sdf[sdf["labels_tuple"] == target_combo]
    filtered_splits[split] = Dataset.from_pandas(filtered_df, preserve_index=False)

from datasets import DatasetDict
filtered_ds_dict = DatasetDict(filtered_splits)
filtered_ds_dict.save_to_disk("rmr1_sft_failure_success_split")
print(filtered_ds_dict)


Saving the dataset (1/1 shards): 100%|██████████| 297/297 [00:00<00:00, 27182.25 examples/s]

DatasetDict({
    train: Dataset({
        features: ['context_messages', 'winner', 'ground_truth', 'labels', 'labels_tuple'],
        num_rows: 1348
    })
    validation: Dataset({
        features: ['context_messages', 'winner', 'ground_truth', 'labels', 'labels_tuple'],
        num_rows: 186
    })
    test: Dataset({
        features: ['context_messages', 'winner', 'ground_truth', 'labels', 'labels_tuple'],
        num_rows: 297
    })
})


In [21]:
import json
from datasets import load_from_disk
import os

# Load your filtered dataset
ds = load_from_disk("rmr1_sft_failure_success_split")

# Define output directory
out_dir = "rmr1_sft_failure_success_split2"
os.makedirs(out_dir, exist_ok=True)

# Save each split to JSONL inside out_dir
for split, split_ds in ds.items():
    out_file = os.path.join(out_dir, f"{split}.jsonl")
    with open(out_file, "w", encoding="utf-8") as f:
        for record in split_ds:
            f.write(json.dumps(record) + "\n")
    print(f"💾 Saved {split} split to {out_file} with {len(split_ds)} examples")


💾 Saved train split to rmr1_sft_failure_success_split2/train.jsonl with 1348 examples
💾 Saved validation split to rmr1_sft_failure_success_split2/validation.jsonl with 186 examples
💾 Saved test split to rmr1_sft_failure_success_split2/test.jsonl with 297 examples


In [ ]:
from huggingface_hub import create_repo

repo_id = "ROOT/rmr1-sft-failure-success-only"
create_repo(repo_id, repo_type="dataset", private=True)

from huggingface_hub import upload_folder

# Single-split dataset
upload_folder(
    repo_id="ROOT/rmr1-sft-failure-success-only",
    folder_path="rmr1_sft_failure_success_split2",
    repo_type="dataset"
)

In [ ]:
example = dataset["train"][0]
print(json.dumps(example, indent=2))


In [ ]:
from datasets import load_from_disk

# Load the dataset from disk
dataset = load_from_disk("rmr1_sft_dataset")
len(dataset)


In [ ]:
example = dataset[0]
example


In [ ]:
# Print user/system messages
for msg in example["context_messages"]:
    print(f"{msg['role'].upper()}: {msg['content']}\n")

# Print selected winner
print("Winner:", example["winner"])


In [ ]:
from datasets import load_from_disk

# Update this path to your actual dataset location
dataset_path = "../rmr1_sft_split_dataset"

# Try to load the dataset
try:
    dataset = load_from_disk(dataset_path)
    print("✅ Dataset loaded successfully.")
    print("📦 Available splits:", dataset.keys())

    test_data = dataset["test"]
    print(f"🔍 Loaded test split with {len(test_data)} examples")
    print("📝 Columns:", test_data.column_names)

    # Show a sample
    print("🔹 Sample example:")
    print(test_data[0])

except Exception as e:
    print("❌ Failed to load dataset:", str(e))


In [ ]:
from huggingface_hub import create_repo, upload_folder

# Upload your local folder
upload_folder(
    repo_id="ROOT/harpa-rlvr",
    folder_path="../model/RM-R1-Qwen2.5-Instruct-7B-RLVR-after-Distill-LR1.0e-6/global_step_26/actor/huggingface",  
    repo_type="model"
)


In [ ]:
from datasets import load_from_disk, DatasetDict, Dataset
import os

# Load dataset
#dataset_path = "../rmr1_sft_split_dataset"
dataset_path = "harpa_rlvr_pref_train_format"
#new_dataset_path = "../rlvr_preference_data"
new_dataset_path = "../rlvr_preference_data_all_success_failure"

try:
    dataset = load_from_disk(dataset_path)
    print("✅ Dataset loaded successfully.")
    print("📦 Available splits:", dataset.keys())

    preference_data_splits = {}

    for split_name in dataset.keys():
        split = dataset[split_name]

        # Select and rename columns
        formatted_split = split.map(lambda example: {
            "context_messages": example["context_messages"],
            "winner": example["ground_truth"]
        })

        preference_data_splits[split_name] = formatted_split

    # Create a new DatasetDict and save
    preference_dataset = DatasetDict(preference_data_splits)
    preference_dataset.save_to_disk(new_dataset_path)
    print(f"💾 New dataset saved at {new_dataset_path}")

except Exception as e:
    print("❌ Failed to load or process dataset:", str(e))

In [ ]:
from datasets import load_from_disk, DatasetDict

dataset_path = "harpa_rlvr_pref_train_format"
new_dataset_path = "../rlvr_preference_data_all_success_failure"

try:
    dataset = load_from_disk(dataset_path)
    print("✅ Dataset loaded successfully.")
    print(dataset)

    # If it's already a DatasetDict, use as-is
    if isinstance(dataset, DatasetDict):
        base_ds = dataset
    else:
        # Wrap single Dataset into a DatasetDict with only "train"
        base_ds = DatasetDict({"train": dataset})

    preference_data_splits = {}

    for split_name in base_ds.keys():
        split = base_ds[split_name]

        # Select and rename columns
        formatted_split = split.map(lambda example: {
            "context_messages": example["context_messages"],
            "winner": example["ground_truth"]
        })

        preference_data_splits[split_name] = formatted_split

    # Create a new DatasetDict and save
    preference_dataset = DatasetDict(preference_data_splits)
    preference_dataset.save_to_disk(new_dataset_path)
    print(f"💾 New dataset saved at {new_dataset_path}")

except Exception as e:
    print("❌ Failed to load or process dataset:", str(e))


In [ ]:
from datasets import load_from_disk

# Load the saved dataset
dataset = load_from_disk("../rlvr_preference_data")

# Check available splits
print("✅ Splits:", dataset.keys())

# Print sample from each split
for split_name in dataset.keys():
    print(f"\n🔍 Example from split: {split_name}")
    print(dataset[split_name][0])


conda create -n verl-merge python=3.11
conda activate verl-merge
cd verl
pip install -e .

python verl/scripts/model_merger.py \
  --local_dir /weka/ROOTv/hypothesis_generation/ASD-specific/model/RM-R1-Qwen2.5-Instruct-7B-RLVR-after-Distill-LR1.0e-6/global_step_26/actor/ \
  --hf_upload_path ROOT/harpa-rlvr


In [7]:
from datasets import load_from_disk

dataset_path = "/weka/harpa-cs/harpa-rlvr-branch/hypothesis_generation/ASD-specific/RM-R1-data/rmr1_sft_failure_success_split"
dataset = load_from_disk(dataset_path)
test_data = dataset["test"]

example = test_data[0]  # or any index
print(example.keys())

def format_messages_as_prompt(messages):
    return "\n".join([f"{m['role']}: {m['content']}" for m in messages])
prompt = format_messages_as_prompt(example["context_messages"])
print(example["winner"])


dict_keys(['context_messages', 'winner', 'ground_truth', 'labels', 'labels_tuple'])
<rubric>
1. Execution Success: 
   - Proposal A: 1.0 - Well-structured with clear implementation steps, appropriate pilot modes, and realistic technical requirements
   - Proposal B: 0.3 - Multiple complex dependencies (OpenAI API, ConceptNet, multi-dataset coordination) with high failure risk

2. Complexity:
   - Proposal A: 0.7 - Moderately complex with iterative refinement and dual approaches but manageable scope
   - Proposal B: 0.9 - Very complex with multiple integrated systems, API dependencies, and cross-domain evaluation

3. Cost Efficiency:
   - Proposal A: 0.8 - Uses local models primarily, minimal API costs, efficient pilot structure
   - Proposal B: 0.2 - Heavy OpenAI API usage for data generation, multiple expensive model calls

4. Expected Hypothesis Validity:
   - Proposal A: 0.8 - Clear testable hypothesis with appropriate baselines and ablation studies
   - Proposal B: 0.4 - Hypothesis

In [ ]:
# Load the Model 
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "ROOT/harpa-rlvr"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="flash_attention_2"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Single Turn Example - from Reward Bench 

conversation = example["context_messages"]

input_ids = tokenizer.apply_chat_template(
    conversation, 
    tokenize=True, 
    add_generation_prompt=True,
    return_tensors="pt").to(model.device)

generation = model.generate(
    input_ids=input_ids,
    max_new_tokens=8192,  # For optimal performance benchmarking, please set this to unlimited (e.g., 50000)
    do_sample=False,
)

completion = tokenizer.decode(
    generation[0][len(input_ids[0]):], 
    skip_special_tokens=True, 
    clean_up_tokenization_spaces=True
)

print(completion)


In [1]:
import torch
from safetensors.torch import save_file
import os, json

# your folder with .pt shard files
pt_dir = "/weka/harpa-cs/harpa-rlvr-branch/hypothesis_generation/ASD-specific/harpa_rlvr_all_success_failure/RM-R1-Qwen2.5-Instruct-7B-RLVR-after-Distill-LR1.0e-6/global_step_1000/actor"
output_folder = "/weka/harpa-cs/harpa-rlvr-branch/hypothesis_generation/ASD-specific/harpa_rlvr_all_success_failure/RM-R1-Qwen2.5-Instruct-7B-RLVR-after-Distill-LR1.0e-6/global_step_1000/actor/huggingface"
import os
import re
import torch
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoConfig, AutoModelForCausalLM
from torch.distributed._tensor import DTensor, Shard, Placement

def merge_by_placement(tensors, placement):
    if placement.is_replicate():
        return tensors[0]
    elif placement.is_partial():
        raise NotImplementedError("Partial placement not supported")
    elif placement.is_shard():
        return torch.cat(tensors, dim=placement.dim).contiguous()
    else:
        raise ValueError(f"Unsupported placement: {placement}")

def merge_fsdp_shards(local_dir, save_dir=None, save_as_safetensors=True):
    # Auto path setup
    assert not local_dir.endswith("huggingface"), "Remove trailing /huggingface"
    save_dir = save_dir or os.path.join(local_dir, "huggingface")

    # Detect world size
    world_size = None
    for f in os.listdir(local_dir):
        match = re.match(r"model_world_size_(\d+)_rank_0\.pt", f)
        if match:
            world_size = int(match.group(1))
            break
    assert world_size is not None, "Could not detect world size"

    # Load rank 0 state dict
    rank_0_path = os.path.join(local_dir, f"model_world_size_{world_size}_rank_0.pt")
    rank_0_state = torch.load(rank_0_path, map_location="cpu")
    pivot_key = sorted(rank_0_state.keys())[0]
    weight = rank_0_state[pivot_key]
    assert isinstance(weight, DTensor)

    mesh = weight.device_mesh.mesh
    mesh_dim_names = weight.device_mesh.mesh_dim_names
    assert mesh_dim_names == ("fsdp",), f"Unsupported mesh: {mesh_dim_names}"

    total_shards = mesh.shape[-1]
    print(f"📦 Detected {total_shards} shards")

    model_state_dict_lst = [None] * total_shards
    model_state_dict_lst[0] = rank_0_state

    def load_rank(rank):
        path = os.path.join(local_dir, f"model_world_size_{world_size}_rank_{rank}.pt")
        model_state_dict_lst[rank] = torch.load(path, map_location="cpu")

    with ThreadPoolExecutor(max_workers=min(32, os.cpu_count())) as executor:
        for rank in range(1, total_shards):
            executor.submit(load_rank, rank)

    state_dict = {}
    param_placements = {}

    for key in model_state_dict_lst[0].keys():
        values = []
        for state in model_state_dict_lst:
            tensor = state[key]
            if isinstance(tensor, DTensor):
                values.append(tensor._local_tensor.bfloat16())
                placements = tuple(tensor.placements)
                param_placements.setdefault(key, placements)
            else:
                values = tensor.bfloat16()
                break
        state_dict[key] = values

    for key in state_dict:
        if isinstance(state_dict[key], list):
            placement = param_placements[key][0]
            state_dict[key] = merge_by_placement(state_dict[key], placement)

    # Save in HF format
    print("💾 Saving Hugging Face format")
    config = AutoConfig.from_pretrained(save_dir)
    model = AutoModelForCausalLM.from_config(config)
    model.save_pretrained(save_dir, state_dict=state_dict, safe_serialization=save_as_safetensors)
    print(f"✅ Model saved at: {save_dir}")

# Example usage (adjust the path)
merge_fsdp_shards(
    local_dir=pt_dir,
    save_dir=output_folder,
    save_as_safetensors=True  # Set False to save as .bin
)


/opt/miniconda3/envs/rm-r1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_2573/3013206734.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weig

📦 Detected 8 shards


/tmp/ipykernel_2573/3013206734.py:59: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_state_dict_lst[rank] = torch.load(path, map_location="cpu")


💾 Saving Hugging Face format
✅ Model saved at: /weka/harpa-cs/harpa-rlvr-branch/hypothesis_generation/ASD-specific/harpa_rlvr_all_success_failure/RM-R1-Qwen2.5-Instruct-7B-RLVR-after-Distill-LR1.0e-6/global_step_1000/actor/huggingface


In [ ]:
from huggingface_hub import create_repo, upload_folder

# create model
create_repo(repo_id="ROOT/harpa-rlvr_all_data_final", repo_type="model", private=True)
# Upload your local folder
upload_folder(
    repo_id="ROOT/harpa-rlvr_all_data_final",
    folder_path="/weka/harpa-cs/harpa-rlvr-branch/hypothesis_generation/ASD-specific/harpa_rlvr_all_success_failure/RM-R1-Qwen2.5-Instruct-7B-RLVR-after-Distill-LR1.0e-6/global_step_1000/actor/huggingface",  
    repo_type="model"
)


In [3]:
import os
import re
import torch
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoConfig, AutoModelForCausalLM
from torch.distributed._tensor import DTensor, Shard, Placement

def merge_by_placement(tensors, placement):
    if placement.is_replicate():
        return tensors[0]
    elif placement.is_partial():
        raise NotImplementedError("Partial placement not supported")
    elif placement.is_shard():
        return torch.cat(tensors, dim=placement.dim).contiguous()
    else:
        raise ValueError(f"Unsupported placement: {placement}")

def merge_fsdp_shards(local_dir, save_dir=None, save_as_safetensors=True):
    # Auto path setup
    assert not local_dir.endswith("huggingface"), "Remove trailing /huggingface"
    save_dir = save_dir or os.path.join(local_dir, "huggingface")

    # Detect world size
    world_size = None
    for f in os.listdir(local_dir):
        match = re.match(r"model_world_size_(\d+)_rank_0\.pt", f)
        if match:
            world_size = int(match.group(1))
            break
    assert world_size is not None, "Could not detect world size"

    # Load rank 0 state dict
    rank_0_path = os.path.join(local_dir, f"model_world_size_{world_size}_rank_0.pt")
    rank_0_state = torch.load(rank_0_path, map_location="cpu")
    pivot_key = sorted(rank_0_state.keys())[0]
    weight = rank_0_state[pivot_key]
    assert isinstance(weight, DTensor)

    mesh = weight.device_mesh.mesh
    mesh_dim_names = weight.device_mesh.mesh_dim_names
    assert mesh_dim_names == ("fsdp",), f"Unsupported mesh: {mesh_dim_names}"

    total_shards = mesh.shape[-1]
    print(f"📦 Detected {total_shards} shards")

    model_state_dict_lst = [None] * total_shards
    model_state_dict_lst[0] = rank_0_state

    def load_rank(rank):
        path = os.path.join(local_dir, f"model_world_size_{world_size}_rank_{rank}.pt")
        model_state_dict_lst[rank] = torch.load(path, map_location="cpu")

    with ThreadPoolExecutor(max_workers=min(32, os.cpu_count())) as executor:
        for rank in range(1, total_shards):
            executor.submit(load_rank, rank)

    state_dict = {}
    param_placements = {}

    for key in model_state_dict_lst[0].keys():
        values = []
        for state in model_state_dict_lst:
            tensor = state[key]
            if isinstance(tensor, DTensor):
                values.append(tensor._local_tensor.bfloat16())
                placements = tuple(tensor.placements)
                param_placements.setdefault(key, placements)
            else:
                values = tensor.bfloat16()
                break
        state_dict[key] = values

    for key in state_dict:
        if isinstance(state_dict[key], list):
            placement = param_placements[key][0]
            state_dict[key] = merge_by_placement(state_dict[key], placement)

    # Save in HF format
    print("💾 Saving Hugging Face format")
    config = AutoConfig.from_pretrained(save_dir)
    model = AutoModelForCausalLM.from_config(config)
    model.save_pretrained(save_dir, state_dict=state_dict, safe_serialization=save_as_safetensors)
    print(f"✅ Model saved at: {save_dir}")

# Example usage (adjust the path)
merge_fsdp_shards(
    local_dir="/weka/harpa-cs/harpa-rlvr-branch/hypothesis_generation/ASD-specific/harpa_rlvr_sep15/RM-R1-Qwen2.5-Instruct-7B-RLVR-after-Distill-LR2.0e-6/global_step_162/actor",
    save_dir="/weka/harpa-cs/harpa-rlvr-branch/hypothesis_generation/ASD-specific/harpa_rlvr_sep15/RM-R1-Qwen2.5-Instruct-7B-RLVR-after-Distill-LR2.0e-6/global_step_162/actor/huggingface",
    save_as_safetensors=True  # Set False to save as .bin
)


/tmp/ipykernel_301749/2976280477.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  rank_0_state = torch.load(rank_0_path, map_location="cpu")


📦 Detected 4 shards


/tmp/ipykernel_301749/2976280477.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_state_dict_lst[rank] = torch.load(path, map_location="cpu")


💾 Saving Hugging Face format
✅ Model saved at: /weka/harpa-cs/harpa-rlvr-branch/hypothesis_generation/ASD-specific/harpa_rlvr_sep15/RM-R1-Qwen2.5-Instruct-7B-RLVR-after-Distill-LR2.0e-6/global_step_162/actor/huggingface


In [ ]:
from huggingface_hub import create_repo, upload_folder

# create model
create_repo(repo_id="ROOT/harpa-rlvr_final2", repo_type="model", private=True)
# Upload your local folder
upload_folder(
    repo_id="ROOT/harpa-rlvr_final2",
    folder_path="/weka/harpa-cs/harpa-rlvr-branch/hypothesis_generation/ASD-specific/harpa_rlvr_sep15/RM-R1-Qwen2.5-Instruct-7B-RLVR-after-Distill-LR2.0e-6/global_step_162/actor/huggingface",  
    repo_type="model"
)


In [1]:
from datasets import load_dataset

# Replace with your actual file path
path = "/weka/harpa-cs/harpa-rlvr-branch/hypothesis_generation/ASD-specific/RM-R1-data/preference_dataset.jsonl"

# Load the jsonl file
dataset = load_dataset("json", data_files=path, split="train")

# Print dataset info
print(dataset)

# Show the first item
print(dataset[0])

# Show all available keys (columns)
print(dataset.column_names)


/opt/miniconda3/envs/rm-r1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 3954 examples [00:01, 2073.94 examples/s]

Dataset({
    features: ['prompt', 'response_a', 'response_b', 'meta_data_response_a', 'meta_data_response_b', 'ground_truth', 'pair_topic', 'labels'],
    num_rows: 3954
})
{'prompt': 'Generate a research hypothesis and proposal given this topic: Information Retrieval', 'response_a': "Here is the proposed Hypothesis: Dynamic sparsification tailored to syntactic complexity of queries and semantic density score of documents will significantly improve retrieval speed and accuracy compared to static sparsification methods.\n\nResearch Proposal:\nDescription: You are an autonomous agent, tasked to perform the following research task:\nTASK DEFINITION:\n================\nName: Dynamic Sparsification for Complex Queries\nShort Description: Dynamic sparsification tailored to syntactic complexity and semantic density improves retrieval speed and accuracy.\nHypothesis to explore: Dynamic sparsification tailored to syntactic complexity of queries and semantic density score of documents will sign